# Etapa 5 — Recuperación Generativa (tipo TIGER) y Análisis Comparativo
## Cuaderno de Reproducibilidad para Google Colab

Este cuaderno permite **verificar de forma liviana** los resultados de la **Etapa 5** de la tesis, sin tener que repetir las ~6 horas de la corrida completa (10 semillas × 15 modelos):

1. Clona la etiqueta del repositorio, instala dependencias y prepara los datos (MovieLens `ml-latest-small` y embeddings `all-MiniLM-L6-v2`).
2. Corre los tests unitarios de la Etapa 5.
3. **Recalcula todas las cifras publicadas** desde los JSON versionados (`verify_report.py`) y comprueba que la semilla 42 coincide con la Fase 2c.
4. **Reentrena y reevalúa la(s) semilla(s) elegida(s)** (por defecto la 42) y compara cada NDCG@10 contra el valor publicado.
5. Muestra las tablas finales de referencia (10 semillas, GPU local).

---
### Modelos de la Etapa 5
- **G1**: recuperación generativa con *Semantic IDs* jerárquicos `[c1, c2, c3, d]` (RQ-VAE sobre embeddings de texto) y un modelo tipo T5 entrenado desde cero, con decodificación por haz restringida a ítems válidos no vistos.
- **G2**: control de G1 con IDs jerárquicos **aleatorios** (misma estructura).
- **G3**: control secuencial con **un token atómico** por película.
- **G1b**: ablación de G1 con embeddings sin el año del título (*no se reproduce en este cuaderno*).
- **Baselines**: popularidad; Two-Tower A/B/C2/D2 y Two-Stage (retrieval A + ranker MLP); y las Etapas 1 y 2 (SVD y NCF) reevaluadas con el mismo protocolo.

**Protocolo**: split temporal por usuario (80% train / 20% test; validación = último 10% de train), relevante = rating ≥ 3.5, evaluación *full-ranking* sobre los ítems no vistos, 592 usuarios de test.

---
> **NOTAS METODOLÓGICAS Y LIMITACIONES**:
> 1. **Entorno de referencia**: las tablas oficiales se registraron en una GPU local **NVIDIA RTX 4060 Laptop 8 GB** (PyTorch `2.6.0+cu124`, Python 3.11, Windows 11). Colab asigna otra GPU y otras versiones de librerías.
> 2. **Diferencias esperables en Colab**: el generador aleatorio de la GPU y los kernels CUDA difieren, por lo que **no se espera coincidencia bit a bit**. El criterio de este cuaderno es que la diferencia con el valor publicado quede dentro de ~2 desvíos entre semillas (la celda de comparación lo calcula). Una diferencia así es ruido, no una discrepancia.
> 3. **bf16**: los modelos generativos se entrenan con precisión mixta `bfloat16`. Las GPU T4 no soportan bf16 de forma nativa y puede ser más lento o cambiar la numérica; si Colab ofrece una L4 o A100, conviene usarla. La celda de verificación del entorno lo informa.
> 4. **Alcance liviano**: se reproduce una semilla de un subconjunto de modelos. La corrida completa (10 semillas, bootstrap por usuarios, G1b) no se hace acá; ver el README para los comandos.
> 5. **Tiempo estimado**: en la RTX 4060, una semilla del subconjunto por defecto (4 discriminativos + G1, G2 y G3 + SVD/NCF) ronda los 25–30 minutos. Es una estimación a partir de los tiempos locales; **no está medida en Colab**.
> 6. **Interpretación**: los resultados de la Etapa 5 son una comparación entre paradigmas en un dataset chico (610 usuarios), no un ranking de «mejor modelo». Las explicaciones causales que aparecen en el README están marcadas como hipótesis.

---
## 1. Descarga del proyecto y posicionamiento en el directorio
Configura la rama/etiqueta a clonar y las semillas a ejecutar. Por defecto clona la etiqueta `stage5-final` (o la rama activa `05-generative-retrieval`).

In [ ]:
# @title Configuración de descarga del proyecto
GIT_REF = "stage5-final"  #@param {type:"string"}
METODO = "github"  #@param ["github", "upload_zip", "drive_zip"]
REPO_URL = "https://github.com/marcosgoizueta/tesis-03-two-stage.git"  #@param {type:"string"}
DRIVE_ZIP_PATH = "/content/drive/MyDrive/03-two-stage.zip"  #@param {type:"string"}

# Configuración de semillas a ejecutar:
# Por defecto solo semilla 42. Cada semilla adicional suma ~25-30 min (estimación local). Rango válido: 42 a 51
SEEDS = [42]  #@param {type:"raw"}

# Opción para persistir resultados en Google Drive (permite usar --resume si se corta la sesión)
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/recsys_stage5_results"  #@param {type:"string"}

import os, sys, glob, subprocess

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
    print(f"Directorio en Drive activo: {DRIVE_RESULTS_DIR}")

if METODO == "github":
    repo_name = os.path.basename(REPO_URL).replace('.git', '')
    if not os.path.exists(f"/content/{repo_name}"):
        print(f"Clonando {REPO_URL}...")
        !git clone {REPO_URL} /content/{repo_name}
    %cd /content/{repo_name}
    print(f"Haciendo checkout a la referencia '{GIT_REF}'...")
    checkout_res = subprocess.run(["git", "checkout", GIT_REF], capture_output=True, text=True)
    if checkout_res.returncode != 0:
        print("\n" + "!" * 75)
        print(f"⚠️ ERROR: No se pudo hacer checkout a la referencia '{GIT_REF}'.")
        print("Detalle de Git:", checkout_res.stderr.strip() or checkout_res.stdout.strip())
        print("Por favor, verificá que la etiqueta o rama exista en el repositorio remoto.")
        print("Podés cambiar el valor de GIT_REF en el formulario superior (ej. '05-generative-retrieval' o un commit hash).")
        print("!" * 75 + "\n")
        raise RuntimeError(f"Referencia de Git '{GIT_REF}' no encontrada en el repositorio. Verificá la rama o etiqueta.")
    else:
        print(f"✅ Checkout exitoso a '{GIT_REF}'.")

elif METODO == "upload_zip":
    zips = glob.glob("/content/*.zip")
    if not zips:
        raise FileNotFoundError("⚠️ Arrastrá el archivo .zip al panel de archivos de Colab y volvé a correr esta celda.")
    zip_file = zips[0]
    print(f"Descomprimiendo {zip_file}...")
    !unzip -q -o {zip_file} -d /content/03-two-stage
    %cd /content/03-two-stage

elif METODO == "drive_zip":
    if not os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')
    !unzip -q -o {DRIVE_ZIP_PATH} -d /content/03-two-stage
    %cd /content/03-two-stage

# Asegurar que estamos en el directorio con requirements-colab-stage5.txt
if not os.path.exists("requirements-colab-stage5.txt"):
    subdirs = [d for d in glob.glob("**/requirements-colab-stage5.txt", recursive=True)]
    if subdirs:
        target_dir = os.path.dirname(subdirs[0])
        %cd {target_dir}

print("\nDirectorio de trabajo actual:", os.getcwd())
print("Archivos principales encontrados:", [f for f in os.listdir('.') if not f.startswith('.')])

---
## 2. Instalación de dependencias y verificación del entorno (GPU y librerías)
Instala `requirements-colab-stage5.txt` (sin fijar `torch` ni CUDA, para respetar el entorno de Colab) y compara las versiones activas con las del entorno de referencia local.

In [ ]:
!pip install -q -r requirements-colab-stage5.txt

import sys, torch, transformers, numpy, sklearn

print("=" * 70)
print("VERIFICACIÓN DE ACELERADOR GPU Y LIBRERÍAS")
print("=" * 70)

if not torch.cuda.is_available():
    print("\n" + "!" * 70)
    print("⚠️ ADVERTENCIA CRÍTICA: NO SE DETECTÓ GPU ACTIVA EN GOOGLE COLAB.")
    print("El entrenamiento de G1/G2/G3 en CPU sería extremadamente lento.")
    print("Activá la GPU en: Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> T4 GPU (o L4/A100 si hay).")
    print("!" * 70 + "\n")
else:
    print(f"GPU detectada    : {torch.cuda.get_device_name(0)}")
    print(f"CUDA en PyTorch  : {torch.version.cuda}")
    print(f"Capacidad Compute: {torch.cuda.get_device_capability(0)}")
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"bf16 soportado   : {bf16_ok}")
    if not bf16_ok:
        print("ℹ️ Esta GPU no soporta bf16 de forma nativa: el entrenamiento generativo puede ser más lento "
              "o cambiar numéricamente respecto a la referencia. Es esperable en T4.")
    !nvidia-smi

REF = {"python": "3.11.9", "torch": "2.6.0+cu124", "transformers": "4.57.6", "numpy": "1.23.5", "scikit-learn": "1.7.2"}
ACT = {"python": sys.version.split()[0], "torch": torch.__version__, "transformers": transformers.__version__,
       "numpy": numpy.__version__, "scikit-learn": sklearn.__version__}
print("\n--- Versiones activas vs entorno de referencia local ---")
for lib, ref_v in REF.items():
    act_v = ACT[lib]
    print(f"{lib:14}: Activa = {act_v:16} | Referencia = {ref_v:14} -> {'✅ Coincide' if act_v == ref_v else 'ℹ️ Difiere (esperable en Colab)'}")
print("=" * 70)

---
## 3. Datos y embeddings de texto
- **MovieLens `ml-latest-small`**: se descarga automáticamente si falta.
- **Embeddings `all-MiniLM-L6-v2`** (9.724 × 384): por defecto se **copia la matriz de referencia versionada** (`reference/stage4/movie_embeddings_minilm.npy`), que es bit a bit la que usó el autor, para que G1 (cuyo tokenizador RQ-VAE depende de ella) parta de la misma entrada. Con `USE_REFERENCE_EMBEDDINGS = False` se regeneran en Colab y se compara la similitud coseno contra la referencia.

In [ ]:
# @title Datos y embeddings
USE_REFERENCE_EMBEDDINGS = True  #@param {type:"boolean"}

import shutil, numpy as np
from pathlib import Path
sys.path.insert(0, "src")
from recsys.config import Stage3Config
from recsys.data import load_and_prepare_data

cfg = Stage3Config()
base_data = load_and_prepare_data(cfg)          # descarga ml-latest-small si no está
assert Path("data/ml-latest-small/ratings.csv").exists() and Path("data/ml-latest-small/movies.csv").exists()
print("MovieLens listo:", Path("data/ml-latest-small").resolve())

REF_EMB = Path("reference/stage4/movie_embeddings_minilm.npy")
CACHE = Path("data/cache"); CACHE.mkdir(parents=True, exist_ok=True)
dst = CACHE / "movie_embeddings_minilm.npy"

if USE_REFERENCE_EMBEDDINGS:
    assert REF_EMB.exists(), "Falta reference/stage4/movie_embeddings_minilm.npy en el repositorio."
    shutil.copy(REF_EMB, dst)
    print(f"Embeddings de referencia copiadas a {dst}")
else:
    from recsys.stage4.embeddings import generate_or_load_item_embeddings
    embs_colab, _ = generate_or_load_item_embeddings(
        movies_df=base_data.movies, item2idx=base_data.item2idx, cache_dir=CACHE,
        device="cuda" if torch.cuda.is_available() else "cpu")
    embs_ref = np.load(REF_EMB)
    cos = np.sum(embs_colab * embs_ref, axis=1)
    print(f"Similitud coseno media={cos.mean():.8f} | mínima={cos.min():.8f} | dif. abs. máx={np.max(np.abs(embs_colab - embs_ref)):.3e}")
    print("✅ Numéricamente idéntica a la referencia." if cos.mean() > 0.99999 else
          "ℹ️ Leves diferencias numéricas entre entornos; el RQ-VAE de G1 puede variar ligeramente.")

emb = np.load(dst)
assert emb.shape == (9724, 384), emb.shape
print("Embeddings en uso:", emb.shape, emb.dtype)

---
## 4. Tests unitarios de la Etapa 5
Ejecuta las suites de `tests/stage5` (split temporal sin fuga, RQ-VAE, tokenizador, decodificación restringida, modelo T5 y baselines de las Etapas 1-2). Los tests de regresión de las Etapas 3 y 4 no se incluyen acá: dependen de la GPU concreta (ver el cuaderno de la Etapa 4).

In [ ]:
!python -m pytest tests/stage5 -q -p no:cacheprovider

---
## 5. Verificación de las cifras publicadas (sin reentrenar)
`verify_report.py` recalcula media y desvío de cada modelo desde los 85 JSON por semilla (`results/stage5/per_seed/`), los compara contra `multi_seed_summary.json` y comprueba que la semilla 42 coincide con la Fase 2c (G1 0.0658, G2 0.0399 y G3 0.0832 con haz 50). Debe terminar en **PASS**.

In [ ]:
!python scripts/verify_report.py --strict

---
## 6. Reproducción liviana: reentrenar y reevaluar la(s) semilla(s) elegida(s)
Reentrena con las recetas ya publicadas y compara con los resultados oficiales:
- **Discriminativos** (30 épocas fijas): Two-Tower A, C2, D2 y Two-Stage.
- **Generativos** (G1, G2, G3 con haz 50 y G3 con ranking exacto): `E*` fijo (G1/G2: 18 épocas, G3: 16) elegido en validación.
- **Etapas 1 y 2**: SVD, NCF (15 épocas) y NCF con épocas elegidas por validación.

Se pueden apagar grupos para acortar la corrida. Los resultados se guardan en `results/colab/` (o en Drive) y **nunca** pisan los oficiales; si se corta la sesión, volver a ejecutar la celda retoma lo ya hecho.

In [ ]:
# @title Reproducción de la(s) semilla(s)
RUN_DISCRIMINATIVE = True  #@param {type:"boolean"}
RUN_GENERATIVE = True      #@param {type:"boolean"}
RUN_EXPLICIT = True        #@param {type:"boolean"}

import time

COLAB_DIR = DRIVE_RESULTS_DIR if SAVE_TO_DRIVE else "results/colab"
seeds_str = " ".join(map(str, SEEDS))
models = []
if RUN_DISCRIMINATIVE: models += ["FP_Two-Tower_A", "FP_Two-Tower_C2", "FP_Two-Tower_D2", "FP_Two-Stage"]
if RUN_GENERATIVE:     models += ["G1", "G2", "G3_beam", "G3_exact"]
models_str = " ".join(models)

print(f"Semillas: {SEEDS} | Resultados en: {COLAB_DIR}")
t0 = time.time()
if models:
    print(f"\n>>> Modelos: {models_str}")
    !python scripts/run_stage5_save_recs.py --seeds {seeds_str} --models {models_str} --out-dir {COLAB_DIR}/recs
if RUN_EXPLICIT:
    print("\n>>> SVD / NCF / NCF_val")
    !python scripts/run_stage5_explicit_baselines.py --seeds {seeds_str} --out-dir {COLAB_DIR}
elapsed = time.time() - t0
print(f"\nCorrida finalizada. Tiempo total: {int(elapsed // 60)}m {int(elapsed % 60)}s")

---
## 7. Comparación contra los resultados oficiales
Para cada modelo y semilla muestra el NDCG@10 obtenido en Colab, el oficial de la misma semilla y la diferencia, medida en desvíos entre semillas del resultado oficial (10 semillas, GPU local). **Criterio**: diferencia ≤ 2 desvíos → ✅ dentro del ruido esperable; mayor → ⚠️ revisar. Con una sola semilla es una verificación de plausibilidad, no una prueba de equivalencia.

In [ ]:
import json, os, numpy as np, pandas as pd
from IPython.display import display

PUB = "results/stage5"
sens = json.load(open(f"{PUB}/sensitivity_30ep_summary.json"))["protocol_fixed_30ep"]
summ = json.load(open(f"{PUB}/multi_seed_summary.json"))["models_summary"]

def pub_std(name):
    if name.startswith("FP_"):
        return sens[name[3:]]["ndcg@10_std"]
    return summ[name]["ndcg@10_std"]

rows = []
chk_path = f"{COLAB_DIR}/recs/repro_check.json"
if os.path.exists(chk_path):
    for key, v in json.load(open(chk_path)).items():
        name, seed = key.split("|")
        sd = pub_std(name)
        rows.append((name, int(seed), v["ndcg_new"], v["ndcg_published"], sd))
for name in ("SVD", "NCF", "NCF_val"):
    vals = [json.load(open(f"{PUB}/per_seed_explicit/{name}_seed_{s}.json"))["ndcg@10"] for s in range(42, 52)]
    for s in SEEDS:
        f_new = f"{COLAB_DIR}/per_seed_explicit/{name}_seed_{s}.json"
        if os.path.exists(f_new):
            rows.append((name, s, json.load(open(f_new))["ndcg@10"],
                         json.load(open(f"{PUB}/per_seed_explicit/{name}_seed_{s}.json"))["ndcg@10"], float(np.std(vals, ddof=1))))

if rows:
    df = pd.DataFrame(rows, columns=["modelo", "semilla", "NDCG@10 Colab", "NDCG@10 oficial", "desvío oficial (10 sem.)"])
    df["diferencia"] = df["NDCG@10 Colab"] - df["NDCG@10 oficial"]
    df["dif. en desvíos"] = df["diferencia"].abs() / df["desvío oficial (10 sem.)"]
    df["estado"] = np.where(df["diferencia"].abs() < 1e-9, "✅ idéntico",
                    np.where(df["dif. en desvíos"] <= 2, "✅ dentro del ruido", "⚠️ revisar"))
    display(df.round(4).sort_values(["semilla", "modelo"]).reset_index(drop=True))
    n_bad = int((df["estado"] == "⚠️ revisar").sum())
    print(f"\n{len(df)} comparaciones; {n_bad} fuera de ±2 desvíos." if n_bad else f"\n{len(df)} comparaciones, todas dentro del ruido esperable.")
else:
    print("No hay resultados de Colab todavía: ejecutá la sección 6.")

---
## 8. Tablas de referencia oficiales (10 semillas, GPU RTX 4060 local)
Elegí el reporte a mostrar. Todas las cifras salen de JSON versionados en `results/stage5/`; ninguna fue escrita a mano.

In [ ]:
# @title Reporte de referencia
REPORTE = "final_comparison"  #@param ["final_comparison", "bias_diversity", "sensitivity_30ep_analysis", "explicit_baselines_analysis", "g1_exact/g1_exact_analysis"]

from IPython.display import Markdown, display
path = f"results/stage5/{REPORTE}.md"
display(Markdown(open(path, encoding="utf-8").read()))

---
## 9. Cómo leer los resultados y qué no se comprobó
- **Comprobado** (10 semillas): los discriminativos con contenido (Two-Tower C2 y D2) superan a los generativos en este dataset; G3 es indistinguible de popularidad; G1 y G2 quedan por debajo de popularidad; G1 − G2 es una tendencia que no supera la corrección de Holm; Two-Stage supera a A con parada por validación pero no con 30 épocas fijas; SVD y NCF quedan por debajo de popularidad.
- **Hipótesis no probadas**: el efecto de escala (610 usuarios es poco para un modelo tipo TIGER) y el distinto presupuesto de ajuste entre generativos y Two-Tower.
- **Fuera de alcance de este cuaderno**: G1b, las 10 semillas, el bootstrap por usuarios y el análisis de sesgo (necesitan las listas top-50 de todas las corridas, que no se versionan en git).

Para la corrida completa (≈6 h en la RTX 4060) ver los comandos de la sección «Reproducción de la Etapa 5» del README.